In [0]:
from pyspark.sql.functions import col, year, month, quarter, lag, avg, stddev
from pyspark.sql.window import Window

silver_df = spark.read.table("workspace.default.silver_overdose")

display(silver_df)
print(f"Total rows: {silver_df.count()}")

State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,State_Name,Predicted_Value
AK,Cocaine (T40.5),2016-05-01,11.0,100.0,0.0,Alaska,11.0
AK,Cocaine (T40.5),2016-10-01,13.0,100.0,0.06991377301,Alaska,13.0
AK,Cocaine (T40.5),2017-12-01,18.0,100.0,0.0,Alaska,18.0
AK,Cocaine (T40.5),2019-04-01,10.0,100.0,0.0,Alaska,10.0
AK,Cocaine (T40.5),2020-11-01,18.0,100.0,0.0,Alaska,18.0
AK,Cocaine (T40.5),2022-06-01,10.0,100.0,0.0,Alaska,10.0
AK,Cocaine (T40.5),2023-10-01,31.0,100.0,0.0,Alaska,31.0
AK,Cocaine (T40.5),2024-03-01,36.0,100.0,0.0,Alaska,36.0
AK,Cocaine (T40.5),2025-05-01,28.0,100.0,0.0,Alaska,28.0
AK,Cocaine (T40.5),2025-06-01,30.0,100.0,0.0,Alaska,30.0


Total rows: 47365


In [0]:
# create ML features
# these are the same features we used in phase 2:
# year, month, quarter, lag_1, lag_3, rolling mean and std

window_spec = Window.partitionBy("State", "Indicator").orderBy("Date")

gold_df = silver_df \
    .withColumn("year", year(col("Date"))) \
    .withColumn("month", month(col("Date"))) \
    .withColumn("quarter", quarter(col("Date"))) \
    .withColumn("lag_1", lag("Death_Count", 1).over(window_spec)) \
    .withColumn("lag_3", lag("Death_Count", 3).over(window_spec)) \
    .withColumn("roll_mean_3", avg("Death_Count").over(
        window_spec.rowsBetween(-2, 0))) \
    .withColumn("roll_std_3", stddev("Death_Count").over(
        window_spec.rowsBetween(-2, 0))) \
    .dropna(subset=["lag_1", "lag_3", "roll_mean_3", "roll_std_3"])

print(f"Silver rows: {silver_df.count()}")
print(f"Gold rows: {gold_df.count()}")
display(gold_df)

Silver rows: 47365
Gold rows: 45813


State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,State_Name,Predicted_Value,year,month,quarter,lag_1,lag_3,roll_mean_3,roll_std_3
AK,Cocaine (T40.5),2016-07-01,13.0,100.0,0.0,Alaska,13.0,2016,7,3,11.0,11.0,11.666666666666666,1.1547005383792517
AK,Cocaine (T40.5),2016-08-01,12.0,100.0,0.02338634238,Alaska,12.0,2016,8,3,13.0,11.0,12.0,1.0
AK,Cocaine (T40.5),2016-09-01,11.0,100.0,0.0469924812,Alaska,11.0,2016,9,3,12.0,11.0,12.0,1.0
AK,Cocaine (T40.5),2016-10-01,13.0,100.0,0.06991377301,Alaska,13.0,2016,10,4,11.0,13.0,12.0,1.0
AK,Cocaine (T40.5),2016-11-01,15.0,100.0,0.06994637445,Alaska,15.0,2016,11,4,13.0,12.0,13.0,2.0
AK,Cocaine (T40.5),2016-12-01,15.0,100.0,0.06888633754,Alaska,15.0,2016,12,4,15.0,11.0,14.333333333333334,1.1547005383792517
AK,Cocaine (T40.5),2017-01-01,15.0,100.0,0.06858710562,Alaska,15.0,2017,1,1,15.0,13.0,15.0,0.0
AK,Cocaine (T40.5),2017-02-01,18.0,100.0,0.06901311249,Alaska,18.0,2017,2,1,15.0,15.0,16.0,1.7320508075688772
AK,Cocaine (T40.5),2017-03-01,19.0,100.0,0.06882312457,Alaska,19.0,2017,3,1,18.0,15.0,17.333333333333332,2.0816659994661326
AK,Cocaine (T40.5),2017-04-01,16.0,100.0,0.06936416185,Alaska,16.0,2017,4,2,19.0,15.0,17.666666666666668,1.5275252316519465


In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_overdose")

print("Gold table created successfully")
print(f"Columns: {gold_df.columns}")
print(f"Total rows: {gold_df.count()}")

Gold table created successfully
Columns: ['State', 'Indicator', 'Date', 'Death_Count', 'Percent_Complete', 'Percent_Pending_Investigation', 'State_Name', 'Predicted_Value', 'year', 'month', 'quarter', 'lag_1', 'lag_3', 'roll_mean_3', 'roll_std_3']
Total rows: 45813
